# Emotional Support Conversational Model — Training Notebook

Fine-tunes `DialoGPT-medium` on a custom emotional support dataset using LoRA.
All checkpoints are auto-saved to Google Drive so progress is never lost.

**Pipeline:** Mount Drive → Install deps → Load data → Tokenize → Train (LoRA) → Save → Test

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Change this path if you want a different folder
DRIVE_SAVE_DIR = '/content/drive/MyDrive/EmotionalSupportModel'

os.makedirs(f'{DRIVE_SAVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_SAVE_DIR}/final_model', exist_ok=True)
os.makedirs(f'{DRIVE_SAVE_DIR}/final_model_merged', exist_ok=True)
os.makedirs(f'{DRIVE_SAVE_DIR}/dataset', exist_ok=True)

print(f'Google Drive mounted. Saving to: {DRIVE_SAVE_DIR}')

## Step 2 — Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes sentencepiece
print('All dependencies installed.')

## Step 3 — Upload or Load Dataset

In [ ]:
from google.colab import files
import shutil

DATASET_DRIVE_PATH = f'{DRIVE_SAVE_DIR}/dataset/emotional_support_dataset.jsonl'
DATASET_LOCAL_PATH = '/content/emotional_support_dataset.jsonl'

if os.path.exists(DATASET_DRIVE_PATH):
    shutil.copy(DATASET_DRIVE_PATH, DATASET_LOCAL_PATH)
    print('Dataset loaded from Google Drive.')
else:
    print('Upload your emotional_support_dataset.jsonl file:')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.copy(fname, DATASET_LOCAL_PATH)
        shutil.copy(fname, DATASET_DRIVE_PATH)
    print('Dataset uploaded and backed up to Drive.')

## Step 4 — Explore Dataset

In [ ]:
import json

data = []
with open(DATASET_LOCAL_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f'Total conversations: {len(data)}')
print('\n--- Sample ---')
for turn in data[0]['conversation']:
    print(f"[{turn['role'].upper()}]: {turn['content']}")

## Step 5 — Tokenize and Prepare Data

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'microsoft/DialoGPT-medium'
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded.')

def format_conversation(convo):
    text = ''
    for turn in convo:
        text += turn['content'] + tokenizer.eos_token
    return text

formatted_texts = [format_conversation(item['conversation']) for item in data]
print(f'{len(formatted_texts)} conversations formatted.')

In [ ]:
from datasets import Dataset
import pandas as pd

df = pd.DataFrame({'text': formatted_texts})
dataset = Dataset.from_pandas(df)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_eval  = eval_dataset.map(tokenize_function,  batched=True, remove_columns=['text'])
tokenized_train.set_format('torch')
tokenized_eval.set_format('torch')
print('Tokenization complete.')

## Step 6 — Load Model with LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

print(f'GPU available: {torch.cuda.is_available()}')

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['c_attn', 'c_proj'],
    bias='none'
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print('LoRA applied.')

## Step 7 — Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, TrainerCallback

LOCAL_CHECKPOINT_DIR = '/content/checkpoints'
os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=LOCAL_CHECKPOINT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

class DriveSyncCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if state.best_model_checkpoint:
            name = os.path.basename(state.best_model_checkpoint)
            dst  = f'{DRIVE_SAVE_DIR}/checkpoints/{name}'
            shutil.copytree(state.best_model_checkpoint, dst, dirs_exist_ok=True)
            print(f'Checkpoint saved to Drive: {dst}')

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f'Epoch {int(state.epoch)} done | best loss: {state.best_metric}')

print('Training arguments ready.')

## Step 8 — Train

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[DriveSyncCallback()]
)

# Auto-resume from Drive checkpoint if available
drive_ckpts = [
    f for f in os.listdir(f'{DRIVE_SAVE_DIR}/checkpoints')
    if f.startswith('checkpoint-')
]
resume_from = None
if drive_ckpts:
    latest = sorted(drive_ckpts)[-1]
    local_path = f'/content/checkpoints/{latest}'
    shutil.copytree(f'{DRIVE_SAVE_DIR}/checkpoints/{latest}', local_path, dirs_exist_ok=True)
    resume_from = local_path
    print(f'Resuming from checkpoint: {latest}')
else:
    print('Starting fresh training run...')

trainer.train(resume_from_checkpoint=resume_from)
print('Training complete!')

## Step 9 — Save Model to Drive

In [ ]:
LOCAL_MODEL_PATH = '/content/emotional_support_model'

trainer.save_model(LOCAL_MODEL_PATH)
tokenizer.save_pretrained(LOCAL_MODEL_PATH)
shutil.copytree(LOCAL_MODEL_PATH, f'{DRIVE_SAVE_DIR}/final_model', dirs_exist_ok=True)
print(f'Model saved to Drive: {DRIVE_SAVE_DIR}/final_model')

## Step 10 — Merge LoRA Weights (Standalone Model)

In [ ]:
from peft import PeftModel

MERGED_LOCAL_PATH = '/content/emotional_support_model_merged'

base_for_merge = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
peft_model_loaded = PeftModel.from_pretrained(base_for_merge, LOCAL_MODEL_PATH)
merged_model = peft_model_loaded.merge_and_unload()
merged_model.save_pretrained(MERGED_LOCAL_PATH)
tokenizer.save_pretrained(MERGED_LOCAL_PATH)
shutil.copytree(MERGED_LOCAL_PATH, f'{DRIVE_SAVE_DIR}/final_model_merged', dirs_exist_ok=True)
print(f'Merged model saved to Drive: {DRIVE_SAVE_DIR}/final_model_merged')

## Step 11 — Test Inference

In [ ]:
inf_tokenizer = AutoTokenizer.from_pretrained(MERGED_LOCAL_PATH)
inf_model = AutoModelForCausalLM.from_pretrained(
    MERGED_LOCAL_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)
inf_model.eval()

def generate_response(user_input, history=None, max_new_tokens=150):
    context = ''
    if history:
        for h_user, h_bot in history[-3:]:
            context += h_user + inf_tokenizer.eos_token
            context += h_bot  + inf_tokenizer.eos_token
    context += user_input + inf_tokenizer.eos_token
    inputs = inf_tokenizer.encode(context, return_tensors='pt').to(inf_model.device)
    with torch.no_grad():
        output = inf_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.3,
            pad_token_id=inf_tokenizer.eos_token_id,
            eos_token_id=inf_tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs.shape[-1]:]
    return inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

tests = [
    'I feel like nobody understands me.',
    'I lost my job and I do not know what to do.',
    'I feel so alone even when I am around people.',
]
for msg in tests:
    print(f'USER: {msg}')
    print(f'BOT:  {generate_response(msg)}')
    print('-' * 50)

## Step 12 — Download Model as Zip

In [ ]:
import zipfile
from google.colab import files as colab_files

ZIP_PATH = '/content/emotional_support_model_export.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(MERGED_LOCAL_PATH):
        for fname in filenames:
            fpath   = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, MERGED_LOCAL_PATH)
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'Zipped model: {size_mb:.1f} MB')
colab_files.download(ZIP_PATH)